In [ ]:
import pandas as pd
import numpy as np
from sklearn import datasets
from evidently import Report, Dataset, DataDefinition, Regression
from evidently.metrics import MeanError, MAE, MAPE, RMSE, R2Score, AbsMaxError, DummyMAE, DummyMAPE, DummyRMSE
from evidently.presets import DataDriftPreset

# Step 1: Load the dataset
# Replace 'DelayedFlights.csv' with the path to your downloaded dataset
data = pd.read_csv('archive/DelayedFlights.csv')

# Step 2: Preprocess the dataset
# Select relevant features and target (ArrDelay as the target)
features = ['DepTime', 'Distance', 'AirTime']  # Example features, adjust based on dataset
target = 'ArrDelay'

# Drop rows with missing values in selected columns for simplicity
data = data[features + [target]].dropna()

# Add simulated predictions (actual ArrDelay + random noise, similar to model_quality.ipynb)
data['prediction'] = data[target] + np.random.normal(0, 10, data.shape[0])

# Step 3: Create reference and current datasets
# Randomly sample 5000 rows for each (adjust size based on dataset)
reference_data = data.sample(n=5000, replace=False, random_state=42)
current_data = data.sample(n=5000, replace=False, random_state=43)

# Step 4: Define data definition for EvidentlyAI
data_definition = DataDefinition(
    regression=[Regression(target=target, prediction="prediction")]
)

# Step 5: Create EvidentlyAI datasets
reference_dataset = Dataset.from_pandas(
    pd.DataFrame(reference_data),
    data_definition=data_definition
)

current_dataset = Dataset.from_pandas(
    pd.DataFrame(current_data),
    data_definition=data_definition
)

# Step 6: Create a model quality report
regression_report = Report([
    MeanError(),
    MAE(),
    MAPE(),
    RMSE(),
    R2Score(),
    AbsMaxError(),
    DummyMAE(),
    DummyMAPE(),
    DummyRMSE(),
])

# Run the report
regression_snapshot = regression_report.run(current_dataset, reference_dataset)

# Save the report as HTML
regression_snapshot.save_html("flight_delay_model_quality_report.html")

# Step 7: Create a data drift report (optional, to check for distribution shifts)
drift_report = Report([
    DataDriftPreset(method="psi")
], include_tests=True)
drift_eval = drift_report.run(reference_data, current_data)
drift_eval.save_html("flight_delay_data_drift_report.html")

print("Model quality and data drift reports generated successfully!")